# 01 — Inspect dataset joins
Set the three dataset paths below. No dataset or model downloads happen here.
Speaker IDs must resolve to **body appearance IDs on the same page** for this experiment. The code reports observed types and rejects face or identity references. Unresolved/off-page links and ambiguous multi-speaker texts are reported and excluded, without becoming negative labels.

Sources: [Manga109 annotation format](https://manga109.github.io/manga109-project-website/en/annotations.html) and [Manga109Dialog XML format](https://github.com/manga109/public-annotations).


In [ ]:
import sys
from pathlib import Path

ROOT = Path(r"C:\Users\ESHAAN\BYTE\Applied-Task\Magi\Manga109Association")
sys.path.insert(0, str(ROOT))
from common import *

DATA.mkdir(exist_ok=True)
CHECKPOINTS.mkdir(exist_ok=True)


In [ ]:
# Edit these paths to your local dataset folders.
BASE_XML_DIR = Path(r"C:\DATASETS\Manga109\annotations")
IMAGE_DIR = Path(r"C:\DATASETS\Manga109\images")
DIALOG_XML_DIR = Path(r"C:\DATASETS\public-annotations\Manga109Dialog\annotations")

settings = {"base_xml_dir": str(BASE_XML_DIR), "image_dir": str(IMAGE_DIR), "dialog_xml_dir": str(DIALOG_XML_DIR), "seed": SEED}
save_json(DATA / "dataset_settings.json", settings)
pages, speaker_types, issues = load_pages(BASE_XML_DIR, DIALOG_XML_DIR, IMAGE_DIR)
print("Observed speaker references:", speaker_types)
print("Pages:", len(pages), "Books:", len({p["book"] for p in pages}))
print("Usable linked texts:", sum(len(p["speaker_links"]) for p in pages))
print("Excluded relation issues:", len(issues))
save_json(DATA / "join_issues.json", issues)
save_json(DATA / "joined_pages.json", pages)
save_json(DATA / "speaker_semantics.json", {"observed_types": speaker_types, "candidate_object_type": "body", "issues": len(issues)})
print("Example excluded issues:", issues[:10])


## Inspect about 30 pages
Blue boxes are text, orange boxes are body appearances, green arrows are ground truth. Original IDs remain visible. Check image dimensions, boxes, and arrows. Pages are sampled from those with usable links using a fixed seed.


In [ ]:
import random

linked_pages = [p for p in pages if p["speaker_links"]]
assert len(linked_pages) >= 30, "At least 30 linked pages are required for inspection."
inspection_pages = random.Random(SEED).sample(linked_pages, 30)
print("Speaker relation example:", inspection_pages[0]["speaker_links"])
for page in inspection_pages:
    show_page(page)


## Confirm the visual join
After reviewing all 30 outputs, set the boolean below to `True` and run the cell. Notebook 02 requires this saved confirmation. Changing the joined records invalidates it.


In [ ]:
VISUAL_JOIN_LOOKS_CORRECT = False

assert VISUAL_JOIN_LOOKS_CORRECT, "Review the 30 pages first, then set this to True."
save_json(DATA / "inspection_ok.json", {
    "joined_hash": digest(pages), "dataset_settings_hash": digest(settings),
    "pages_reviewed": [[p["book"], p["page_index"]] for p in inspection_pages],
    "speaker_object_type": "body",
})
print("Inspection recorded. Continue with notebook 02.")
